# Long-Tailed Object Detection: YOLOv11 with Re-Weighting

This notebook anchors the baseline training pipeline for drone-view object detection using YOLOv11 **without any pretrained weights**.
Fill in each `# TODO` placeholder before executing the training cell below.


In [1]:
from pathlib import Path
from datetime import datetime
from typing import List, Tuple, Optional
import random
import shutil

from PIL import Image
import yaml

PROJECT_DIR = Path.cwd()
DATA_ROOT = (PROJECT_DIR / '../../../dataset/taica-cvpdl-2025-hw-2/CVPDL_hw2/CVPDL_hw2').resolve()
TRAIN_DIR = DATA_ROOT / 'train'
TEST_DIR = DATA_ROOT / 'test'

if not TRAIN_DIR.exists():
    raise FileNotFoundError(f'Dataset train folder not found at {TRAIN_DIR}. Update DATA_ROOT if your layout differs.')

if not TEST_DIR.exists():
    print(f'Warning: test folder not found at {TEST_DIR}. Test images will not be copied.')
    TEST_DIR = None

EXPERIMENT_NAME = 'yolov11_rebalance_test'
RUN_ID = datetime.now().strftime('%Y%m%d-%H%M%S')
EXPERIMENT_DIR = (PROJECT_DIR / 'artifacts' / EXPERIMENT_NAME / RUN_ID).resolve()
YOLO_DATA_DIR = EXPERIMENT_DIR / 'dataset'
RUN_ROOT_DIR = EXPERIMENT_DIR / 'run'
TRAIN_RUN_NAME = 'train'
VAL_RUN_NAME = 'val'
INFER_RUN_NAME = 'infer'

EXPERIMENT_DIR.mkdir(parents=True, exist_ok=True)
RUN_ROOT_DIR.mkdir(parents=True, exist_ok=True)

VAL_RATIO = 0.2  # fixed 80/20 train/val split
SEED = 11


def load_label_file(label_path: Path) -> List[Tuple[int, float, float, float, float]]:
    boxes: List[Tuple[int, float, float, float, float]] = []
    with label_path.open('r') as fh:
        for raw_line in fh:
            line = raw_line.strip()
            if not line:
                continue
            parts = [p.strip() for p in line.split(',')]
            if len(parts) < 5:
                continue
            cls = int(parts[0])
            x, y, w, h = map(float, parts[1:5])
            boxes.append((cls, x, y, w, h))
    return boxes


def convert_tlwh_to_yolo(x: float, y: float, w: float, h: float, img_w: int, img_h: int) -> Tuple[float, float, float, float]:
    xc = (x + w / 2.0) / img_w
    yc = (y + h / 2.0) / img_h
    ww = w / img_w
    hh = h / img_h
    xc = min(max(xc, 0.0), 1.0)
    yc = min(max(yc, 0.0), 1.0)
    ww = min(max(ww, 0.0), 1.0)
    hh = min(max(hh, 0.0), 1.0)
    return xc, yc, ww, hh


def prepare_dataset(train_dir: Path, output_dir: Path, val_ratio: float, seed: int, test_dir: Optional[Path] = None):
    if output_dir.exists():
        shutil.rmtree(output_dir)
    images_root = output_dir / 'images'
    labels_root = output_dir / 'labels'
    for split in ['train', 'val']:
        (images_root / split).mkdir(parents=True, exist_ok=True)
        (labels_root / split).mkdir(parents=True, exist_ok=True)
    if test_dir is not None:
        (images_root / 'test').mkdir(parents=True, exist_ok=True)
        (labels_root / 'test').mkdir(parents=True, exist_ok=True)

    image_stems = sorted({path.stem for path in train_dir.glob('*.png')})
    missing_labels: List[str] = []
    missing_images: List[str] = []

    for label_path in train_dir.glob('*.txt'):
        if not (train_dir / f'{label_path.stem}.png').exists():
            missing_images.append(label_path.stem)

    records: List[Tuple[str, Path, Path]] = []
    for stem in image_stems:
        image_path = train_dir / f'{stem}.png'
        label_path = train_dir / f'{stem}.txt'
        if not label_path.exists():
            missing_labels.append(stem)
            continue
        records.append((stem, image_path, label_path))

    rng = random.Random(seed)
    rng.shuffle(records)

    if not records:
        raise RuntimeError('No image/label pairs found in the training dataset.')

    if len(records) == 1:
        val_count = 0
    else:
        val_count = max(1, int(len(records) * val_ratio))
        val_count = min(len(records) - 1, val_count)

    val_stems = {stem for stem, *_ in records[:val_count]}

    stats = {
        'train': 0,
        'val': 0,
        'test': 0,
        'boxes': 0,
        'missing_labels': missing_labels,
        'missing_images': missing_images,
        'class_ids': set(),
    }

    for stem, image_path, label_path in records:
        split = 'val' if stem in val_stems else 'train'
        with Image.open(image_path) as img:
            img_w, img_h = img.size

        label_entries = load_label_file(label_path)
        yolo_lines = []
        for cls, x, y, w, h in label_entries:
            xc, yc, ww, hh = convert_tlwh_to_yolo(x, y, w, h, img_w, img_h)
            yolo_lines.append(f'{cls} {xc:.6f} {yc:.6f} {ww:.6f} {hh:.6f}')
            stats['class_ids'].add(cls)

        dst_img = images_root / split / f'{stem}.png'
        shutil.copy2(image_path, dst_img)

        dst_label = labels_root / split / f'{stem}.txt'
        dst_label.write_text('\n'.join(yolo_lines))

        stats[split] += 1
        stats['boxes'] += len(yolo_lines)

    if test_dir is not None:
        test_images = sorted(test_dir.glob('*.png'))
        for image_path in test_images:
            shutil.copy2(image_path, images_root / 'test' / image_path.name)
            stats['test'] += 1

    stats['val_ratio'] = val_ratio
    return stats


stats = prepare_dataset(
    TRAIN_DIR,
    YOLO_DATA_DIR,
    val_ratio=VAL_RATIO,
    seed=SEED,
    test_dir=TEST_DIR,
)

print(f'Dataset prepared at {YOLO_DATA_DIR}')
print(f"train images: {stats['train']} | val images: {stats['val']} | boxes: {stats['boxes']}")
if stats['test']:
    print(f"test images copied: {stats['test']}")
if stats['missing_labels']:
    sample = ', '.join(stats['missing_labels'][:5])
    print(f"Skipped {len(stats['missing_labels'])} images without labels. Examples: {sample}")
if stats['missing_images']:
    sample = ', '.join(stats['missing_images'][:5])
    print(f"Skipped {len(stats['missing_images'])} labels without images. Examples: {sample}")

class_ids = sorted(stats['class_ids'])
if not class_ids:
    raise RuntimeError('No class ids found in annotations. Check label parsing logic.')

ALL_CLASS_NAMES = {
    0: 'car',
    1: 'hov',
    2: 'person',
    3: 'motorcycle',
}
missing_class_ids = sorted(set(class_ids) - set(ALL_CLASS_NAMES))
if missing_class_ids:
    raise ValueError(f'Unknown class ids {missing_class_ids} detected. Update ALL_CLASS_NAMES mapping to include them.')

CLASS_NAME_MAP = {cid: ALL_CLASS_NAMES[cid] for cid in class_ids}
DATA_CONFIG_PATH = EXPERIMENT_DIR / 'longtail_dataset.yaml'

data_yaml = {
    'path': YOLO_DATA_DIR.as_posix(),
    'train': 'images/train',
    'val': 'images/val',
}
if stats['test']:
    data_yaml['test'] = 'images/test'
data_yaml['nc'] = len(CLASS_NAME_MAP)
data_yaml['names'] = {cid: name for cid, name in CLASS_NAME_MAP.items()}

with DATA_CONFIG_PATH.open('w') as fh:
    yaml.safe_dump(data_yaml, fh, sort_keys=False)

print(f'YOLO data config saved to {DATA_CONFIG_PATH}')




Dataset prepared at /home/daniel/Study/ComputerVision/2025-CVPDL/HW2/hw2_314706007/code_314706007/src/artifacts/yolov11_rebalance_test/20251028-110716/dataset
train images: 760 | val images: 190 | boxes: 33331
test images copied: 550
YOLO data config saved to /home/daniel/Study/ComputerVision/2025-CVPDL/HW2/hw2_314706007/code_314706007/src/artifacts/yolov11_rebalance_test/20251028-110716/longtail_dataset.yaml


In [2]:
from collections import Counter
import math


def load_yolo_label_entries(label_path: Path) -> list[tuple[int, float, float, float, float]]:
    entries = []
    if not label_path.exists():
        return entries
    for raw in label_path.read_text().strip().splitlines():
        raw = raw.strip()
        if not raw:
            continue
        parts = raw.split()
        if len(parts) < 5:
            continue
        cls = int(parts[0])
        x, y, w, h = map(float, parts[1:5])
        entries.append((cls, x, y, w, h))
    return entries


train_images_dir = YOLO_DATA_DIR / 'images/train'
train_labels_dir = YOLO_DATA_DIR / 'labels/train'
val_labels_dir = YOLO_DATA_DIR / 'labels/val'

if not train_labels_dir.exists():
    raise FileNotFoundError(f'Train labels directory not found at {train_labels_dir}')

TRAIN_RECORDS: list[dict] = []
CLASS_BOX_COUNTS: Counter[int] = Counter()
CLASS_IMAGE_COUNTS: Counter[int] = Counter()
TOTAL_BOXES = 0

for label_path in sorted(train_labels_dir.glob('*.txt')):
    stem = label_path.stem
    boxes = load_yolo_label_entries(label_path)
    class_hist = Counter(box[0] for box in boxes)
    if not class_hist:
        continue
    TRAIN_RECORDS.append(
        {
            'stem': stem,
            'image_path': train_images_dir / f'{stem}.png',
            'label_path': label_path,
            'class_hist': class_hist,
            'box_count': sum(class_hist.values()),
        }
    )
    CLASS_BOX_COUNTS.update(class_hist)
    CLASS_IMAGE_COUNTS.update(class_hist.keys())
    TOTAL_BOXES += sum(class_hist.values())

NUM_CLASSES = len(CLASS_NAME_MAP)
if NUM_CLASSES == 0:
    raise RuntimeError('CLASS_NAME_MAP is empty; dataset preparation may have failed.')

max_count = max(CLASS_BOX_COUNTS.values())
min_count = min(CLASS_BOX_COUNTS.values())
IMBALANCE_RATIO = max_count / max(1, min_count)

print('Class distribution (box counts):')
for cls_id in sorted(CLASS_NAME_MAP):
    name = CLASS_NAME_MAP[cls_id]
    box_count = CLASS_BOX_COUNTS.get(cls_id, 0)
    image_count = CLASS_IMAGE_COUNTS.get(cls_id, 0)
    print(f'  class {cls_id} ({name:>10}): boxes={box_count:5d}, images_with_class={image_count:4d}')

print(f'Total boxes: {TOTAL_BOXES}')
print(f'Imbalance ratio (max/min): {IMBALANCE_RATIO:.3f}')

BASE_DATASET_DIR = YOLO_DATA_DIR
BASE_DATASET_CLASS_COUNTS = CLASS_BOX_COUNTS.copy()
BASE_IMBALANCE_RATIO = IMBALANCE_RATIO


Class distribution (box counts):
  class 0 (       car): boxes=19023, images_with_class= 746
  class 1 (       hov): boxes= 1118, images_with_class= 462
  class 2 (    person): boxes= 2732, images_with_class= 404
  class 3 (motorcycle): boxes= 4264, images_with_class= 400
Total boxes: 27137
Imbalance ratio (max/min): 17.015


In [3]:
from __future__ import annotations

import random
from dataclasses import dataclass
from pathlib import Path
from typing import Optional

import numpy as np
import torch
from ultralytics import YOLO
try:
    import multiprocessing as mp
    mp.set_start_method('fork', force=True)
except RuntimeError:
    pass



@dataclass
class TrainConfig:
    """Configuration container for baseline YOLOv11 training."""

    # TODO: choose the YOLOv11 architecture (Ultralytics bundle names such as 'yolov11n.yaml')
    model_yaml: str = 'yolo11s.yaml'

    # Dataset YAML generated in the preparation cell above
    dataset_yaml: Path = DATA_CONFIG_PATH

    # TODO: adjust batch size based on GPU memory
    batch_size: int = 16

    # TODO: adjust epoch count based on convergence needs
    epochs: int = 100

    image_size: int = 640

    # Use CUDA if available; override with a specific device string if needed (e.g. "0", "0,1", "cpu")
    device: str = 'auto'

    workers: int = 8
    project_dir: Path = RUN_ROOT_DIR
    run_name: str = TRAIN_RUN_NAME
    val_name: str = VAL_RUN_NAME
    seed: int = SEED

    # Optimization hyperparameters for a plain baseline run
    optimizer: str = 'SGD'
    learning_rate: float = 0.01
    final_lr_ratio: float = 0.01  # ratio between final and initial LR (Ultralytics uses cosine by default)
    weight_decay: float = 5e-4
    momentum: float = 0.937
    warmup_epochs: float = 3.0

    # Early stopping patience in epochs; adjust if you need longer training
    patience: int = 50

    # Optional: resume from an earlier run (leave as None for a fresh start)
    resume_checkpoint: Optional[Path] = None


def set_deterministic(seed: int) -> None:
    """Set random seeds for reproducible training."""

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def validate_paths(config: TrainConfig) -> None:
    """Ensure critical input files exist before launching training."""

    if '#TODO' in config.model_yaml:
        raise ValueError('Update TrainConfig.model_yaml with the YOLOv11 architecture name or YAML path you intend to use.')

    dataset_yaml_path = Path(config.dataset_yaml)
    if not dataset_yaml_path.exists():
        raise FileNotFoundError(f'Dataset YAML not found at: {dataset_yaml_path}')

    if config.resume_checkpoint is not None:
        resume_path = Path(config.resume_checkpoint)
        if not resume_path.exists():
            raise FileNotFoundError(f'Resume checkpoint not found at: {resume_path}')


In [4]:
from pathlib import Path
from typing import Any, Dict, Optional


def build_model(config: TrainConfig, initial_weights: Optional[Path] = None) -> YOLO:
    """Instantiate a YOLOv11 model without loading pretrained weights."""
    if initial_weights is not None:
        model = YOLO(str(initial_weights))
    else:
        model = YOLO(config.model_yaml)
    model.overrides['pretrained'] = False
    return model


def build_train_kwargs(config: TrainConfig, extra_kwargs: Optional[Dict[str, Any]] = None) -> Dict[str, Any]:
    """Generate the argument dictionary passed into `YOLO.train`."""
    kwargs: Dict[str, Any] = {
        'data': str(config.dataset_yaml),
        'epochs': config.epochs,
        'batch': config.batch_size,
        'imgsz': config.image_size,
        'device': config.device,
        'workers': config.workers,
        'project': str(config.project_dir),
        'name': config.run_name,
        'exist_ok': True,
        'optimizer': config.optimizer,
        'lr0': config.learning_rate,
        'lrf': config.final_lr_ratio,
        'weight_decay': config.weight_decay,
        'momentum': config.momentum,
        'warmup_epochs': config.warmup_epochs,
        'patience': config.patience,
        'resume': str(config.resume_checkpoint) if config.resume_checkpoint else False,
        'pretrained': False,
        'save_period': -1,
        'verbose': True,
    }
    if extra_kwargs:
        kwargs.update(extra_kwargs)
    return kwargs


def train_single_stage(
    config: TrainConfig,
    *,
    class_weights: Optional[Dict[int, float]] = None,
    extra_train_kwargs: Optional[Dict[str, Any]] = None,
    initial_weights: Optional[Path] = None,
) -> Dict[str, Any]:
    """Train a single stage given a `TrainConfig` and optional overrides."""

    set_deterministic(config.seed)
    validate_paths(config)
    config.project_dir.mkdir(parents=True, exist_ok=True)

    if config.device == 'auto':
        resolved_device = 'cuda' if torch.cuda.is_available() else 'cpu'
        print(f"Auto-selected device: {resolved_device}")
        config.device = resolved_device

    model = build_model(config, initial_weights=initial_weights)
    train_kwargs = build_train_kwargs(config, extra_train_kwargs)
    # Override class-specific parameters (for rebalancing)
    if class_weights is not None:
        model.overrides['cls'] = 2.0

    print(f"Using {train_kwargs.get('workers', 'unknown')} dataloader workers for training")

    if class_weights is not None:
        weight_vector = torch.tensor([class_weights.get(cls, 1.0) for cls in range(len(CLASS_NAME_MAP))], dtype=torch.float32)

        def _apply_class_weights(trainer):
            device = getattr(trainer, 'device', None)
            if device is None:
                device = 'cuda' if torch.cuda.is_available() else 'cpu'
            if not isinstance(device, torch.device):
                device = torch.device(device) if isinstance(device, str) else torch.device('cpu')
            weights_device = weight_vector.to(device)
            if hasattr(trainer, 'loss') and hasattr(trainer.loss, 'class_weights'):
                trainer.loss.class_weights = weights_device
            if hasattr(trainer, 'validator') and hasattr(trainer.validator, 'loss_cls') and hasattr(trainer.validator.loss_cls, 'class_weights'):
                trainer.validator.loss_cls.class_weights = weights_device

        model.add_callback('on_train_start', _apply_class_weights)
        model.add_callback('on_train_epoch_start', _apply_class_weights)
        model.add_callback('on_val_start', _apply_class_weights)

    train_result = model.train(**train_kwargs)

    run_dir = Path(getattr(train_result, 'save_dir', config.project_dir / config.run_name))
    best_weights_path = run_dir / 'weights' / 'best.pt'
    if best_weights_path.exists():
        print(f'Best checkpoint available at: {best_weights_path}')
    else:
        print(f'Warning: best checkpoint not found at {best_weights_path}')

    return {
        'model': model,
        'train_result': train_result,
        'run_dir': run_dir,
        'best_weights': best_weights_path,
        'config': config,
    }


In [5]:
from pathlib import Path
from typing import Any, Dict, Optional


def run_validation(
    model: YOLO,
    config: TrainConfig,
    *,
    extra_val_kwargs: Optional[Dict[str, Any]] = None,
) -> Dict[str, Any]:
    """Run validation for a trained YOLO model and capture artifact paths."""

    val_kwargs: Dict[str, Any] = {
        'data': str(config.dataset_yaml),
        'imgsz': config.image_size,
        'batch': config.batch_size,
        'device': config.device,
        'workers': 0,
        'split': 'val',
        'save_json': False,
        'plots': True,
        'verbose': True,
        'project': str(config.project_dir),
        'name': config.val_name,
        'exist_ok': True,
    }
    if extra_val_kwargs:
        val_kwargs.update(extra_val_kwargs)
    val_kwargs['workers'] = 0

    val_results = model.val(**val_kwargs)
    val_dir = Path(val_results.save_dir)
    print(f"Validation metrics saved under: {val_dir}")

    return {
        'val_results': val_results,
        'val_dir': val_dir,
    }


In [6]:
import csv
from pathlib import Path
from typing import Dict, List, Optional


def run_inference_and_build_submission(
    best_weights: Path,
    *,
    test_images_dir: Path,
    project_dir: Path,
    run_name: str,
    image_size: int,
    device: str,
    conf: float = 0.25,
) -> Dict[str, Path]:
    """Run Ultralytics inference and export a Kaggle-style submission CSV."""

    if not best_weights.exists():
        raise FileNotFoundError(f'Best checkpoint not found at {best_weights}')
    if not test_images_dir.exists():
        raise FileNotFoundError(f'Test images folder not found at {test_images_dir}')

    inference_model = YOLO(str(best_weights))
    predict_results = inference_model.predict(
        source=str(test_images_dir),
        project=str(project_dir),
        name=run_name,
        imgsz=image_size,
        device=device,
        conf=conf,
        save=True,
        save_txt=True,
        save_conf=True,
        exist_ok=True,
    )

    if not predict_results:
        raise RuntimeError('No predictions were returned by YOLO inference.')

    prediction_save_dir = Path(predict_results[0].save_dir)
    print(f'Inference outputs saved to: {prediction_save_dir}')

    submission_rows: List[List[str]] = []
    for result in predict_results:
        stem = Path(result.path).stem
        digits = ''.join(ch for ch in stem if ch.isdigit())
        image_id = str(int(digits)) if digits else stem
        image_h, image_w = result.orig_shape
        boxes = result.boxes
        if boxes is None or len(boxes) == 0:
            pred_string = ''
        else:
            xyxy = boxes.xyxy.cpu().numpy()
            confs = boxes.conf.cpu().numpy()
            classes = boxes.cls.cpu().numpy()
            parts: List[str] = []
            for conf, (x1, y1, x2, y2), cls in zip(confs, xyxy, classes):
                x1 = max(0.0, float(x1))
                y1 = max(0.0, float(y1))
                x2 = min(float(x2), float(image_w))
                y2 = min(float(y2), float(image_h))
                width = max(0.0, x2 - x1)
                height = max(0.0, y2 - y1)
                parts.append(f'{conf:.6f} {x1:.2f} {y1:.2f} {width:.2f} {height:.2f} {int(cls)}')
            pred_string = ' '.join(parts)
        submission_rows.append([image_id, pred_string])

    submission_rows.sort(key=lambda row: int(row[0]) if row[0].isdigit() else row[0])

    submission_path = project_dir / f'{run_name}_submission.csv'
    with submission_path.open('w', newline='') as fh:
        writer = csv.writer(fh)
        writer.writerow(['Image_ID', 'PredictionString'])
        writer.writerows(submission_rows)

    print(f'Submission file written to: {submission_path}')

    return {
        'prediction_dir': prediction_save_dir,
        'submission_path': submission_path,
    }


In [7]:
import math
import random
import shutil
from collections import Counter
from dataclasses import dataclass
from typing import Dict, Iterable, List, Optional, Tuple


def summarize_class_counts(records: Iterable[dict]) -> Counter[int]:
    counter: Counter[int] = Counter()
    for rec in records:
        counter.update(rec['class_hist'])
    return counter


def oversample_records(records: List[dict], target_count: Optional[int] = None, seed: int = SEED) -> Tuple[List[dict], Counter[int]]:
    """Duplicate minority samples until each class reaches ``target_count`` boxes."""

    base_counts = summarize_class_counts(records)
    if not base_counts:
        return records, Counter()
    if target_count is None:
        target_count = max(base_counts.values())

    rng = random.Random(seed)
    class_to_records = {cls: [rec for rec in records if rec['class_hist'].get(cls, 0) > 0] for cls in base_counts}
    augmented: List[dict] = list(records)
    duplicate_tracker: Dict[str, int] = {}
    current_counts = Counter(base_counts)

    progress = True
    while progress:
        progress = False
        for cls in sorted(class_to_records):
            if current_counts[cls] >= target_count:
                continue
            candidates = class_to_records[cls]
            if not candidates:
                continue
            rec = rng.choice(candidates)
            suffix = duplicate_tracker.get(rec['stem'], 0)
            duplicate_tracker[rec['stem']] = suffix + 1
            alias = f"{rec['stem']}_dup{suffix:04d}"
            augmented.append({**rec, 'alias': alias})
            for c, n in rec['class_hist'].items():
                current_counts[c] += n
            progress = True
    return augmented, current_counts


def undersample_records(records: List[dict], target_count: Optional[int] = None, seed: int = SEED) -> Tuple[List[dict], Counter[int]]:
    """Remove majority-only samples until reaching ``target_count`` boxes for the majority class."""

    base_counts = summarize_class_counts(records)
    if not base_counts:
        return records, Counter()
    majority_cls = max(base_counts, key=base_counts.get)
    if target_count is None:
        target_count = max(min(base_counts.values()), int(sum(base_counts.values()) / len(base_counts)))

    rng = random.Random(seed)
    retained = list(records)
    current_counts = Counter(base_counts)
    candidates = [rec for rec in retained if set(rec['class_hist'].keys()) == {majority_cls}]
    rng.shuffle(candidates)

    for rec in candidates:
        if current_counts[majority_cls] <= target_count:
            break
        retained.remove(rec)
        for cls, num in rec['class_hist'].items():
            current_counts[cls] -= num
    return retained, current_counts


def apply_resampling(records: List[dict], *, seed: int = SEED, oversample: bool = True, undersample: bool = True) -> Tuple[List[dict], Counter[int]]:
    """Convenience wrapper that applies under/over-sampling sequentially."""

    working_records = list(records)
    current_counts = summarize_class_counts(working_records)

    if undersample:
        working_records, current_counts = undersample_records(working_records, seed=seed)
    if oversample:
        target = max(current_counts.values()) if current_counts else None
        working_records, current_counts = oversample_records(working_records, target_count=target, seed=seed)
    return working_records, current_counts


def compute_class_weights(class_counts: Counter[int], *, method: str = 'effective_num', beta: float = 0.999) -> Dict[int, float]:
    """Derive re-weighting coefficients given class counts."""

    weights: Dict[int, float] = {}
    for cls, count in class_counts.items():
        if count <= 0:
            weights[cls] = 1.0
            continue
        if method == 'inverse':
            weights[cls] = 1.0 / float(count)
        elif method == 'sqrt_inv':
            weights[cls] = 1.0 / math.sqrt(float(count))
        else:  # effective number of samples
            weights[cls] = (1.0 - beta) / (1.0 - beta ** count)
    # Normalise so that mean weight is 1.0
    if weights:
        scale = len(weights) / sum(weights.values())
        for cls in weights:
            weights[cls] *= scale
    return weights


def materialize_dataset(records: List[dict], base_dataset_dir: Path, output_dataset_dir: Path) -> Counter[int]:
    """Create a YOLO dataset at ``output_dataset_dir`` using ``records`` for the training split."""

    if output_dataset_dir.exists():
        shutil.rmtree(output_dataset_dir)

    # Copy validation and test splits verbatim for fair comparison.
    for split in ['val', 'test']:
        src_images = base_dataset_dir / 'images' / split
        src_labels = base_dataset_dir / 'labels' / split
        dst_images = output_dataset_dir / 'images' / split
        dst_labels = output_dataset_dir / 'labels' / split
        if src_images.exists():
            shutil.copytree(src_images, dst_images, dirs_exist_ok=True)
        else:
            dst_images.mkdir(parents=True, exist_ok=True)
        if src_labels.exists():
            shutil.copytree(src_labels, dst_labels, dirs_exist_ok=True)
        else:
            dst_labels.mkdir(parents=True, exist_ok=True)

    train_images_dst = output_dataset_dir / 'images' / 'train'
    train_labels_dst = output_dataset_dir / 'labels' / 'train'
    train_images_dst.mkdir(parents=True, exist_ok=True)
    train_labels_dst.mkdir(parents=True, exist_ok=True)

    class_counter = Counter()
    for rec in records:
        alias = rec.get('alias', rec['stem'])
        shutil.copy2(rec['image_path'], train_images_dst / f'{alias}.png')
        shutil.copy2(rec['label_path'], train_labels_dst / f'{alias}.txt')
        class_counter.update(rec['class_hist'])

    return class_counter


def copy_base_dataset(base_dataset_dir: Path, output_dataset_dir: Path) -> Counter[int]:
    """Clone the baseline dataset without altering the class distribution."""

    if output_dataset_dir.exists():
        shutil.rmtree(output_dataset_dir)
    shutil.copytree(base_dataset_dir, output_dataset_dir)
    return summarize_class_counts(TRAIN_RECORDS)


def write_dataset_yaml(dataset_dir: Path, yaml_path: Path) -> None:
    data_yaml = {
        'path': dataset_dir.as_posix(),
        'train': 'images/train',
        'val': 'images/val',
        'nc': len(CLASS_NAME_MAP),
        'names': {int(k): v for k, v in CLASS_NAME_MAP.items()},
    }
    test_dir = dataset_dir / 'images' / 'test'
    if test_dir.exists() and any(test_dir.iterdir()):
        data_yaml['test'] = 'images/test'
    with yaml_path.open('w') as fh:
        yaml.safe_dump(data_yaml, fh, sort_keys=False)


In [8]:
from dataclasses import dataclass
from datetime import datetime
from typing import Any, Dict, Optional


def read_final_training_metrics(run_dir: Path) -> Dict[str, float]:
    """Load the last row of Ultralytics' results.csv as a metrics dictionary."""

    metrics_path = run_dir / 'results.csv'
    if not metrics_path.exists():
        return {}
    import csv

    with metrics_path.open('r', newline='') as fh:
        reader = csv.DictReader(fh)
        rows = list(reader)
    if not rows:
        return {}
    final_row = rows[-1]
    metrics: Dict[str, float] = {}
    for key, value in final_row.items():
        if key == 'epoch':
            continue
        try:
            metrics[key] = float(value)
        except (TypeError, ValueError):
            continue
    return metrics


@dataclass
class ExperimentScenario:
    name: str
    description: str
    apply_resampling: bool = False
    apply_reweighting: bool = False
    two_stage: bool = False
    epochs: int = 8
    stage1_epochs: int = 4
    stage2_epochs: int = 4
    freeze_backbone_layers: Optional[int] = 10
    seed: int = SEED


def prepare_scenario_datasets(scenario: ExperimentScenario, scenario_dir: Path) -> Dict[str, Any]:
    dataset_info: Dict[str, Any] = {}

    if scenario.two_stage:
        stage1_dir = scenario_dir / 'dataset_stage1'
        if scenario.apply_resampling:
            stage1_records, stage1_counts = apply_resampling(TRAIN_RECORDS, seed=scenario.seed)
            class_counts_stage1 = materialize_dataset(stage1_records, BASE_DATASET_DIR, stage1_dir)
        else:
            class_counts_stage1 = copy_base_dataset(BASE_DATASET_DIR, stage1_dir)
        stage1_yaml = scenario_dir / 'longtail_dataset_stage1.yaml'
        write_dataset_yaml(stage1_dir, stage1_yaml)

        stage2_dir = scenario_dir / 'dataset_stage2'
        if scenario.apply_resampling:
            stage2_records, stage2_counts = apply_resampling(TRAIN_RECORDS, seed=scenario.seed)
            class_counts_stage2 = materialize_dataset(stage2_records, BASE_DATASET_DIR, stage2_dir)
        else:
            class_counts_stage2 = copy_base_dataset(BASE_DATASET_DIR, stage2_dir)
        stage2_yaml = scenario_dir / 'longtail_dataset_stage2.yaml'
        write_dataset_yaml(stage2_dir, stage2_yaml)

        dataset_info['stage1'] = {
            'dataset_dir': stage1_dir,
            'dataset_yaml': stage1_yaml,
            'class_counts': class_counts_stage1,
        }
        dataset_info['stage2'] = {
            'dataset_dir': stage2_dir,
            'dataset_yaml': stage2_yaml,
            'class_counts': class_counts_stage2,
        }
    else:
        dataset_dir = scenario_dir / 'dataset'
        if scenario.apply_resampling:
            records, _ = apply_resampling(TRAIN_RECORDS, seed=scenario.seed)
            class_counts = materialize_dataset(records, BASE_DATASET_DIR, dataset_dir)
        else:
            class_counts = copy_base_dataset(BASE_DATASET_DIR, dataset_dir)
        dataset_yaml = scenario_dir / 'longtail_dataset.yaml'
        write_dataset_yaml(dataset_dir, dataset_yaml)
        dataset_info['primary'] = {
            'dataset_dir': dataset_dir,
            'dataset_yaml': dataset_yaml,
            'class_counts': class_counts,
        }
    return dataset_info


def run_scenario(scenario: ExperimentScenario) -> Dict[str, Any]:
    print(f"=== Running scenario: {scenario.name} ===")
    timestamp = datetime.now().strftime('%Y%m%d-%H%M%S')
    scenario_dir = (PROJECT_DIR / 'artifacts' / scenario.name / timestamp).resolve()
    run_root_dir = scenario_dir / 'run'
    run_root_dir.mkdir(parents=True, exist_ok=True)

    dataset_info = prepare_scenario_datasets(scenario, scenario_dir)

    base_class_counts = BASE_DATASET_CLASS_COUNTS
    class_weights = compute_class_weights(base_class_counts) if scenario.apply_reweighting else None

    summary: Dict[str, Any] = {
        'scenario': scenario.name,
        'description': scenario.description,
        'timestamp': timestamp,
        'scenario_dir': scenario_dir,
        'class_weights': class_weights,
    }

    if scenario.two_stage:
        stage1 = dataset_info['stage1']
        stage2 = dataset_info['stage2']

        stage1_config = TrainConfig(
            workers=0,
            dataset_yaml=stage1['dataset_yaml'],
            batch_size=12,
            epochs=scenario.stage1_epochs,
            project_dir=run_root_dir,
            run_name='train_stage1',
            val_name='val_stage1',
            seed=scenario.seed,
        )
        train_stage1 = train_single_stage(
            stage1_config,
            class_weights=None,
            extra_train_kwargs=None,
        )
        run_validation(train_stage1['model'], stage1_config)

        stage2_config = TrainConfig(
            workers=0,
            dataset_yaml=stage2['dataset_yaml'],
            batch_size=12,
            epochs=scenario.stage2_epochs,
            project_dir=run_root_dir,
            run_name='train_stage2',
            val_name='val_stage2',
            seed=scenario.seed,
        )
        stage2_extra_kwargs = {'freeze': scenario.freeze_backbone_layers} if scenario.freeze_backbone_layers else None
        train_stage2 = train_single_stage(
            stage2_config,
            class_weights=class_weights,
            extra_train_kwargs=stage2_extra_kwargs,
            initial_weights=train_stage1['best_weights'],
        )
        run_validation(train_stage2['model'], stage2_config)

        final_best = train_stage2['best_weights'] if train_stage2['best_weights'].exists() else train_stage1['best_weights']
        final_device = stage2_config.device

        summary['stage1_metrics'] = read_final_training_metrics(train_stage1['run_dir'])
        summary['stage2_metrics'] = read_final_training_metrics(train_stage2['run_dir'])
        summary['final_metrics'] = summary['stage2_metrics'] or summary['stage1_metrics']
        summary['stage1_class_counts'] = dict(stage1['class_counts'])
        summary['stage2_class_counts'] = dict(stage2['class_counts'])
    else:
        primary = dataset_info['primary']
        primary_config = TrainConfig(
            workers=0,
            dataset_yaml=primary['dataset_yaml'],
            batch_size=12,
            epochs=scenario.epochs,
            project_dir=run_root_dir,
            run_name='train',
            val_name='val',
            seed=scenario.seed,
        )
        train_primary = train_single_stage(
            primary_config,
            class_weights=class_weights,
        )
        run_validation(train_primary['model'], primary_config)

        final_best = train_primary['best_weights']
        final_device = primary_config.device
        summary['final_metrics'] = read_final_training_metrics(train_primary['run_dir'])
        summary['class_counts'] = dict(primary['class_counts'])

    inference_info = run_inference_and_build_submission(
        final_best,
        test_images_dir=BASE_DATASET_DIR / 'images' / 'test',
        project_dir=run_root_dir,
        run_name='infer',
        image_size=640,
        device=final_device,
    )

    summary.update(inference_info)
    return summary


SCENARIOS = [
    ExperimentScenario(
        name='yolov11_resampling',
        description='Hybrid over/under-sampling without class re-weighting',
        apply_resampling=True,
        apply_reweighting=False,
    ),
    ExperimentScenario(
        name='yolov11_reweighting',
        description='Original dataset with class re-weighting only',
        apply_resampling=False,
        apply_reweighting=True,
    ),
    ExperimentScenario(
        name='yolov11_reweighting_both',
        description='Resampling plus class re-weighting across the full model',
        apply_resampling=True,
        apply_reweighting=True,
    ),
    ExperimentScenario(
        name='yolov11_reweighting_classifier',
        description='Two-stage training: uniform full-model pretrain then frozen-backbone, class-weighted head',
        apply_resampling=True,
        apply_reweighting=True,
        two_stage=True,
        stage1_epochs=4,
        stage2_epochs=4,
        freeze_backbone_layers=10,
    ),
]





In [9]:
import pandas as pd
import matplotlib.pyplot as plt

EXPERIMENT_SUMMARIES: list[dict] = []
for scenario in SCENARIOS:
    summary = run_scenario(scenario)
    EXPERIMENT_SUMMARIES.append(summary)

comparison_rows = []
for summary in EXPERIMENT_SUMMARIES:
    metrics = summary.get('final_metrics', {})
    map50 = metrics.get('metrics/mAP50(B)', float('nan'))
    map5095 = metrics.get('metrics/mAP50-95(B)', float('nan'))
    precision = metrics.get('metrics/precision(B)', float('nan'))
    recall = metrics.get('metrics/recall(B)', float('nan'))
    comparison_rows.append(
        {
            'scenario': summary['scenario'],
            'description': summary['description'],
            'mAP50': map50,
            'mAP50-95': map5095,
            'precision': precision,
            'recall': recall,
            'submission': str(summary['submission_path']),
        }
    )

results_df = pd.DataFrame(comparison_rows)
print('Validation summary:')
display(results_df)

plt.figure(figsize=(8, 4))
plt.bar(results_df['scenario'], results_df['mAP50'], color='#4C72B0')
plt.ylabel('mAP@0.50')
plt.title('Scenario comparison (validation mAP50)')
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

print('Submission files:')
for summary in EXPERIMENT_SUMMARIES:
    print(f"  {summary['scenario']}: {summary['submission_path']}")


=== Running scenario: yolov11_resampling ===
Auto-selected device: cuda
Using 0 dataloader workers for training
New https://pypi.org/project/ultralytics/8.3.221 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.209 🚀 Python-3.11.13 torch-2.8.0+cu128 CUDA:0 (NVIDIA GeForce RTX 3080, 10240MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=12, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/daniel/Study/ComputerVision/2025-CVPDL/HW2/hw2_314706007/code_314706007/src/artifacts/yolov11_resampling/20251028-110734/longtail_dataset.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=8, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=

,scenario,description,mAP50,mAP50-95,precision,recall,submission
0,yolov11_resampling,Hybrid over/under-sampling without class re-we...,0.41762,0.13634,0.56193,0.40645,/home/daniel/Study/ComputerVision/2025-CVPDL/H...
1,yolov11_reweighting,Original dataset with class re-weighting only,0.04406,0.01028,0.58407,0.08999,/home/daniel/Study/ComputerVision/2025-CVPDL/H...
2,yolov11_reweighting_both,Resampling plus class re-weighting across the ...,0.40060,0.12430,0.58035,0.40620,/home/daniel/Study/ComputerVision/2025-CVPDL/H...
3,yolov11_reweighting_classifier,Two-stage training: uniform full-model pretrai...,0.34865,0.10912,0.52772,0.34621,/home/daniel/Study/ComputerVision/2025-CVPDL/H...


<Figure size 800x400 with 1 Axes>

Submission files:
  yolov11_resampling: /home/daniel/Study/ComputerVision/2025-CVPDL/HW2/hw2_314706007/code_314706007/src/artifacts/yolov11_resampling/20251028-110734/run/infer_submission.csv
  yolov11_reweighting: /home/daniel/Study/ComputerVision/2025-CVPDL/HW2/hw2_314706007/code_314706007/src/artifacts/yolov11_reweighting/20251028-120527/run/infer_submission.csv
  yolov11_reweighting_both: /home/daniel/Study/ComputerVision/2025-CVPDL/HW2/hw2_314706007/code_314706007/src/artifacts/yolov11_reweighting_both/20251028-121447/run/infer_submission.csv
  yolov11_reweighting_classifier: /home/daniel/Study/ComputerVision/2025-CVPDL/HW2/hw2_314706007/code_314706007/src/artifacts/yolov11_reweighting_classifier/20251028-131252/run/infer_submission.csv
